# UCS761 - Deep Learning Lab 3
### Logistic Regression on the Glass Dataset

Taking the sigmoid idea from Lab 2 and building a properly trainable logistic regression on real data.

The dataset has glass samples with various chemical measurements. We are turning this into a binary classification: Type 1 glass vs everything else.

Loss function: binary cross-entropy (not MSE, because this is classification).

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## Step 1: Load the Data

Always look at what you are working with before touching anything else.

Which column is the output: Type
Are all columns numeric: Yes
Is there an ID column to drop: No, all columns are actual features

In [15]:
col_names = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "Type"]
df = pd.read_csv("glass.csv", header=None, names=col_names)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("First 5 rows:")
df.head()

Shape: (214, 10)
Columns: ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
First 5 rows:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
1,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
2,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
3,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
4,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
5,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [16]:
print("Basic info:")
print(df.dtypes)
print("Any nulls:", df.isnull().sum().sum())
print("Class distribution:")
print(df["Type"].value_counts())


Basic info:
RI      float64
Na      float64
Mg      float64
Al      float64
Si      float64
K       float64
Ca      float64
Ba      float64
Fe      float64
Type      int64
dtype: object
Any nulls: 0
Class distribution:
Type
2    76
1    70
7    29
3    17
5    13
6     9
Name: count, dtype: int64


## Step 2: Create Binary Labels

There are multiple glass types but we only want to detect Type 1 vs the rest.

Type 1 gives y = 1 (the class we care about)
Everything else gives y = 0

In [17]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])

print("Label distribution after binarizing:")
print(df["y"].value_counts())


Label distribution after binarizing:
y
0    144
1     70
Name: count, dtype: int64


## Step 3: Separate Features and Labels

In [18]:
X = df.drop(columns=["y"]).values
y = df["y"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive class (Type 1):", y.sum(), "samples")


X shape: (214, 9)
y shape: (214,)
Positive class (Type 1): 70 samples


## Step 4: Train-Test Split

We need to evaluate on data the model has not seen. Testing on training data gives fake confidence since the model could just memorize.

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape[0], "samples")
print("Test: ", X_test.shape[0], "samples")


Train: 171 samples
Test:  43 samples


## Step 5: Scale the Features

The glass dataset features have very different ranges. Without scaling, sigmoid can saturate and gradients become tiny, so learning basically stops.

Fit the scaler on train data only, then apply to both. Never fit on test data.

In [20]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"After scaling - train mean (approx): {X_train.mean():.4f}")
print(f"After scaling - train std  (approx): {X_train.std():.4f}")


After scaling - train mean (approx): 0.0000
After scaling - train std  (approx): 1.0000


## Step 6: Sigmoid and Forward Pass

Same sigmoid from Lab 2. We clip z to avoid overflow for extreme values.

In [21]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def predict_proba(X, w, b):
    z = X @ w + b
    return sigmoid(z)


## Step 7: Binary Cross-Entropy Loss

For classification we use cross-entropy, not MSE.

The key difference: cross-entropy heavily penalizes confident wrong predictions. If the model says 95% probability and it is wrong, the penalty is huge. MSE would not care nearly as much.

L = -mean(y * log(p) + (1-y) * log(1-p))

In [22]:
def bce_loss(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


## Step 8: Weight Update

Gradient of binary cross-entropy with sigmoid comes out clean:

dL/dw = (1/N) * X^T @ (p - y)
dL/db = mean(p - y)

This is standard gradient descent. Move weights in the direction that reduces loss.

In [23]:
def update(X, y, w, b, lr):
    p     = predict_proba(X, w, b)
    err   = p - y
    w     = w - lr * (X.T @ err) / len(y)
    b     = b - lr * np.mean(err)
    return w, b


## Step 9: Training

In [24]:
w      = np.zeros(X_train.shape[1])
b      = 0.0
lr     = 0.1
epochs = 100

loss_history = []

for i in range(epochs):
    w, b = update(X_train, y_train, w, b, lr)
    l    = bce_loss(y_train, predict_proba(X_train, w, b))
    loss_history.append(l)
    if (i + 1) % 20 == 0:
        print(f"Epoch {i+1:3d}  Loss: {l:.4f}")

print(f"Final loss: {loss_history[-1]:.4f}")


Epoch  20  Loss: 0.5776
Epoch  40  Loss: 0.5392
Epoch  60  Loss: 0.5192
Epoch  80  Loss: 0.5066
Epoch 100  Loss: 0.4978
Final loss: 0.4978


## Step 10: Evaluate with Two Thresholds

In [25]:
p_test = predict_proba(X_test, w, b)

pred_50 = (p_test >= 0.5).astype(int)
pred_70 = (p_test >= 0.7).astype(int)

acc_50 = np.mean(pred_50 == y_test)
acc_70 = np.mean(pred_70 == y_test)

print(f"Accuracy at threshold 0.5: {acc_50*100:.2f}%")
print(f"Accuracy at threshold 0.7: {acc_70*100:.2f}%")


Accuracy at threshold 0.5: 86.05%
Accuracy at threshold 0.7: 72.09%


In [26]:
print("Predictions comparison (first 15 test samples):")
print(f"{'p_hat':>8}  {'pred@0.5':>10}  {'pred@0.7':>10}  {'actual':>8}")
for p, p5, p7, actual in list(zip(p_test, pred_50, pred_70, y_test))[:15]:
    print(f"  {p:.4f}       {p5}           {p7}        {actual}")


Predictions comparison (first 15 test samples):
   p_hat    pred@0.5    pred@0.7    actual
  0.4660       0           0        1
  0.0270       0           0        0
  0.6077       1           0        1
  0.0223       0           0        0
  0.3088       0           0        0
  0.3007       0           0        0
  0.5507       1           0        1
  0.4420       0           0        0
  0.4384       0           0        0
  0.4064       0           0        0
  0.0410       0           0        0
  0.0826       0           0        0
  0.5423       1           0        0
  0.4040       0           0        0
  0.2035       0           0        0


## Analysis

**How this differs from the perceptron:**
The perceptron used a step function. Hard 0 or 1, no confidence measure, update rule only fired when a mistake was made.
Logistic regression uses sigmoid so the output is a probability. Cross-entropy measures how wrong the confidence is, not just whether the label was correct. Gradients flow even when predictions are technically right but uncertain.

**Why sigmoid matters:**
Without it, the raw z value can be anything from minus infinity to plus infinity. Sigmoid squashes it into (0,1) so we can treat the output as a probability.

**Why higher threshold is safer for glass QC:**
Accepting a bad glass piece costs more than rejecting a good one. Threshold 0.7 means the model needs to be 70% or more confident before accepting. Fewer false acceptances, which is what you want in quality control.

**What is still missing:**
This model can only learn a linear boundary. If glass types are not linearly separable, accuracy will plateau no matter how long you train. Hidden layers fix this and that is what the later labs cover.